In [1]:
import pandas as pd
import ast
import numpy as np
import os # To help construct paths

# --- Define File Paths ---
# Adjust these paths based on where your files are located
# Assumes the structure shown in your printout
rounakbanik_path = '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7'
grouplens_path = '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1'

links_ml_path = os.path.join(grouplens_path, 'links.csv')
movies_ml_path = os.path.join(grouplens_path, 'movies.csv')
ratings_ml_path = os.path.join(grouplens_path, 'ratings.csv') # Not added to movies_full, but useful later
tags_ml_path = os.path.join(grouplens_path, 'tags.csv') # Not added to movies_full, but useful later
genome_scores_path = os.path.join(grouplens_path, 'genome-scores.csv')
genome_tags_path = os.path.join(grouplens_path, 'genome-tags.csv')

links_tmdb_path = os.path.join(rounakbanik_path, 'links.csv') # Can use either links file
metadata_tmdb_path = os.path.join(rounakbanik_path, 'movies_metadata.csv')
credits_tmdb_path = os.path.join(rounakbanik_path, 'credits.csv')
keywords_tmdb_path = os.path.join(rounakbanik_path, 'keywords.csv')
ratings_tmdb_path = os.path.join(rounakbanik_path, 'ratings.csv') # Not added to movies_full

# --- Load DataFrames ---
print("Loading dataframes...")
try:
    links_ml = pd.read_csv(links_ml_path)
    movies_ml = pd.read_csv(movies_ml_path)
    metadata_tmdb = pd.read_csv(metadata_tmdb_path, low_memory=False) # low_memory=False recommended for this large CSV
    credits_tmdb = pd.read_csv(credits_tmdb_path)
    keywords_tmdb = pd.read_csv(keywords_tmdb_path)
    genome_scores = pd.read_csv(genome_scores_path)
    genome_tags = pd.read_csv(genome_tags_path)
    # Optional: Load ratings if needed later for recommender
    # ratings_ml = pd.read_csv(ratings_ml_path)
    # ratings_tmdb = pd.read_csv(ratings_tmdb_path) # Note: These rating files might be identical or overlap significantly.
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Please check your file paths.")
    exit() # Exit if files aren't found

print("Dataframes loaded.")

# --- Initial Merges (Similar to your code) ---
print("Performing initial merges...")
# Ensure tmdbId in links_ml is numeric and non-null
links_ml = links_ml[links_ml['tmdbId'].notnull()].copy()
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Ensure id in metadata_tmdb is numeric and non-null, rename to tmdbId
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()].copy()
metadata_tmdb['id'] = pd.to_numeric(metadata_tmdb['id'], errors='coerce').astype(int)
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})

# Merge MovieLens movies with links to get tmdbId
movies_full = pd.merge(movies_ml, links_ml, on='movieId', how='inner')
print(f"Merged MovieLens movies and links. Shape: {movies_full.shape}")

# --- Enhance metadata_tmdb Processing ---
print("Processing metadata_tmdb...")

# Convert list-like string columns to actual lists
list_cols = ['genres', 'production_companies', 'production_countries', 'spoken_languages']
for col in list_cols:
    metadata_tmdb[col] = metadata_tmdb[col].fillna('[]').apply(ast.literal_eval)

# Extract primary genre (you already did this)
metadata_tmdb['main_genre'] = metadata_tmdb['genres'].apply(lambda x: x[0]['name'] if isinstance(x, list) and x else None)

# Extract names from list columns
def extract_names(list_of_dicts):
    if isinstance(list_of_dicts, list):
        return [d['name'] for d in list_of_dicts if isinstance(d, dict) and 'name' in d]
    return [] # Return empty list for NaNs or incorrect format

metadata_tmdb['genres_tmdb_list'] = metadata_tmdb['genres'].apply(extract_names)
metadata_tmdb['production_companies_list'] = metadata_tmdb['production_companies'].apply(extract_names)
metadata_tmdb['production_countries_list'] = metadata_tmdb['production_countries'].apply(extract_names)
metadata_tmdb['spoken_languages_list'] = metadata_tmdb['spoken_languages'].apply(extract_names)

# Handle numeric columns: budget, revenue, runtime, vote_average, vote_count
numeric_cols = ['budget', 'revenue', 'runtime', 'vote_average', 'vote_count']
for col in numeric_cols:
    # Convert to numeric, coercing errors to NaN, then fill NaN with 0 (or another appropriate strategy)
    metadata_tmdb[col] = pd.to_numeric(metadata_tmdb[col], errors='coerce').fillna(0)

# Handle release_date
metadata_tmdb['release_date'] = pd.to_datetime(metadata_tmdb['release_date'], errors='coerce')
metadata_tmdb['release_year'] = metadata_tmdb['release_date'].dt.year # Extract year

# Select relevant columns from metadata_tmdb to merge
metadata_cols_to_merge = ['tmdbId', 'budget', 'revenue', 'runtime', 'release_date',
                          'release_year', 'vote_average', 'vote_count', 'overview', 'tagline',
                          'main_genre', 'genres_tmdb_list', 'production_companies_list',
                          'production_countries_list', 'spoken_languages_list', 'status']

metadata_tmdb_subset = metadata_tmdb[metadata_cols_to_merge]

# Merge enhanced metadata
movies_full = pd.merge(movies_full, metadata_tmdb_subset, on='tmdbId', how='left')
print(f"Merged enhanced metadata. Shape: {movies_full.shape}")

# --- Enhance credits_tmdb Processing ---
print("Processing credits_tmdb...")

# Ensure 'cast' and 'crew' are actual lists
credits_tmdb['cast'] = credits_tmdb['cast'].fillna('[]').apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].fillna('[]').apply(ast.literal_eval)

# Extract Director (your function is good)
def get_director(crew):
    if isinstance(crew, list):
        for person in crew:
            if isinstance(person, dict) and person.get('job') == 'Director':
                return person.get('name')
    return None

# Extract Top N Actors
def get_top_actors(cast, n=3):
    if isinstance(cast, list):
        actor_names = [person.get('name') for person in cast if isinstance(person, dict) and 'name' in person]
        return actor_names[:n] # Return the first N names
    return []

credits_tmdb['tmdbId'] = credits_tmdb['id'] # Already done by user

credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['top_actors'] = credits_tmdb['cast'].apply(get_top_actors)

# Select relevant columns from credits_tmdb to merge
credits_cols_to_merge = ['tmdbId', 'director', 'top_actors']

credits_tmdb_subset = credits_tmdb[credits_cols_to_merge]

# Merge enhanced credits
movies_full = pd.merge(movies_full, credits_tmdb_subset, on='tmdbId', how='left')
print(f"Merged credits (director, top actors). Shape: {movies_full.shape}")

# --- Process keywords_tmdb ---
print("Processing keywords_tmdb...")
# Your original processing is good: convert string list to list of names
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords_list'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict) and 'name' in d])

# Select relevant columns and merge
keywords_cols_to_merge = ['id', 'keywords_list']
keywords_tmdb_subset = keywords_tmdb[keywords_cols_to_merge].rename(columns={'id': 'tmdbId'})

movies_full = pd.merge(movies_full, keywords_tmdb_subset, on='tmdbId', how='left')
print(f"Merged keywords. Shape: {movies_full.shape}")

# --- Prepare Genome Data (Keep Separate) ---
print("Processing Genome data...")
# Merge genome scores with genome tags to get tag names
genome_data = pd.merge(genome_scores, genome_tags, on='tagId', how='inner')
print(f"Genome data shape: {genome_data.shape}")
# This DataFrame links movieId to tag name and relevance score.
# Example columns: 'movieId', 'tagId', 'relevance', 'tag'
# It's best kept separate but linked to movies_full by 'movieId'.
# For content-based features, you'd typically use this data structure directly
# to create vectors for movies based on tag relevances.

# --- Finalize movies_full DataFrame ---
# Clean up / rename columns if necessary
movies_full = movies_full.rename(columns={
    'title_x': 'title_ml',       # Original MovieLens title
    'genres_x': 'genres_ml',     # Original MovieLens genres string
    # The TMDB 'genres' column was processed into 'genres_tmdb_list' and 'main_genre'
    'overview': 'overview_tmdb', # Use suffix for clarity
    'tagline': 'tagline_tmdb',   # Use suffix for clarity
    'status': 'status_tmdb'      # Use suffix for clarity
})

# Reorder columns for better readability (optional)
# Define a preferred order if you like, or inspect the columns.
# print("\nFinal columns in movies_full:")
# print(movies_full.columns.tolist())





Loading dataframes...
Dataframes loaded.
Performing initial merges...
Merged MovieLens movies and links. Shape: (57917, 5)
Processing metadata_tmdb...
Merged enhanced metadata. Shape: (57979, 20)
Processing credits_tmdb...
Merged credits (director, top actors). Shape: (58123, 22)
Processing keywords_tmdb...
Merged keywords. Shape: (59365, 23)
Processing Genome data...
Genome data shape: (14862528, 4)


In [3]:
movies_full

,movieId,title,genres,imdbId,tmdbId,budget,revenue,runtime,release_date,release_year,...,tagline_tmdb,main_genre,genres_tmdb_list,production_companies_list,production_countries_list,spoken_languages_list,status_tmdb,director,top_actors,keywords_list
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,30000000.0,373554033.0,81.0,1995-10-30,1995.0,...,NaN,Animation,"[Animation, Comedy, Family]",[Pixar Animation Studios],[United States of America],[English],Released,John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles]","[jealousy, toy, boy, friendship, friends, riva..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,65000000.0,262797249.0,104.0,1995-12-15,1995.0,...,Roll the dice and unleash the excitement!,Adventure,"[Adventure, Fantasy, Family]","[TriStar Pictures, Teitler Film, Interscope Co...",[United States of America],"[English, Français]",Released,Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]","[board game, disappearance, based on children'..."
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,0.0,0.0,101.0,1995-12-22,1995.0,...,Still Yelling. Still Fighting. Still Ready for...,Romance,"[Romance, Comedy]","[Warner Bros., Lancaster Gate]",[United States of America],[English],Released,Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret]","[fishing, best friend, duringcreditsstinger, o..."
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,16000000.0,81452156.0,127.0,1995-12-22,1995.0,...,Friends are the people who let you be yourself...,Comedy,"[Comedy, Drama, Romance]",[Twentieth Century Fox Film Corporation],[United States of America],[English],Released,Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devine]","[based on novel, interracial relationship, sin..."
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,0.0,76578911.0,106.0,1995-02-10,1995.0,...,Just When His World Is Back To Normal... He's ...,Comedy,[Comedy],"[Sandollar Productions, Touchstone Pictures]",[United States of America],[English],Released,Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short]","[baby, midlife crisis, confidence, aging, daug..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59360,193876,The Great Glinka (1946),(no genres listed),38566,78251,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59361,193878,Les tribulations d'une caissière (2011),Comedy,1754787,87558,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59362,193880,Her Name Was Mumu (2016),Drama,5847740,422666,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59363,193882,Flora (2017),Adventure|Drama|Horror|Sci-Fi,4453756,454439,NaN,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("\nGenome data (genome_data) is also loaded and ready, linked by 'movieId'.")
print("Sample of genome_data:")
print(genome_data.head())


Genome data (genome_data) is also loaded and ready, linked by 'movieId'.
Sample of genome_data:
   movieId  tagId  relevance           tag
0        1      1    0.02900           007
1        1      2    0.02375  007 (series)
2        1      3    0.05425  18th century
3        1      4    0.06875         1920s
4        1      5    0.16000         1930s
